# CVG-E: 7B Model Experiments on GPU

This notebook runs the full CVG-E evaluation pipeline on LLaVA-1.5-7B using a cloud GPU.
It evaluates Vanilla, CVG (dual-pass), and CVG-E (hook-based single-pass) across POPE benchmarks.

**Requirements:** GPU runtime (A100 recommended, T4 minimum). ~15GB VRAM for 7B in fp16.

In [ ]:
# Cell 1: Check GPU and install dependencies
!nvidia-smi
!pip install -q torch torchvision transformers accelerate pillow tqdm bitsandbytes

In [ ]:
# Cell 2: Clone repo (or upload files)
import os
REPO_URL = "https://github.com/gauriiiiiiiiiiii/CVG_faithful_mllm"
if not os.path.exists("CVG_faithful_mllm"):
    !git clone {REPO_URL}
os.chdir("CVG_faithful_mllm")
!ls

In [ ]:
# Cell 3: Download COCO val2014 images + POPE questions
!mkdir -p data/coco/val2014
!mkdir -p data/pope

# Download COCO val2014 (if not already present)
if not os.path.exists("data/coco/val2014/COCO_val2014_000000000042.jpg"):
    print("Downloading COCO val2014 images...")
    !wget -q http://images.cocodataset.org/zips/val2014.zip -O /tmp/val2014.zip
    !unzip -q /tmp/val2014.zip -d data/coco/
    !rm /tmp/val2014.zip
    print(f"Downloaded {len(os.listdir('data/coco/val2014'))} images")
else:
    print(f"COCO val2014 already present: {len(os.listdir('data/coco/val2014'))} images")

# Download POPE questions
POPE_BASE = "https://raw.githubusercontent.com/AoiDragon/POPE/main/output/coco"
for split in ["random", "popular", "adversarial"]:
    fname = f"coco_pope_{split}.json"
    if not os.path.exists(f"data/pope/{fname}"):
        !wget -q {POPE_BASE}/{fname} -O data/pope/{fname}
        print(f"Downloaded {fname}")

In [ ]:
# Cell 4: Quick sanity check - load model and test one question
import sys
sys.path.insert(0, ".")

import torch
from PIL import Image
from cvg.modules import ABLATIONS
from cvg.model import CVGModel

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

# Load vanilla model for sanity check
model = CVGModel(
    model_name="llava-hf/llava-1.5-7b-hf",
    config=ABLATIONS["vanilla"],
    dtype=torch.float16,
    use_flash=True,
)

# Test with one image
test_imgs = [f for f in os.listdir("data/coco/val2014") if f.endswith(".jpg")][:1]
img = Image.open(f"data/coco/val2014/{test_imgs[0]}").convert("RGB")
result = model.generate_vanilla(img, "USER: Is there a person in the image?\nASSISTANT:", max_new_tokens=10)
print(f"\nSanity check: {result.text} ({result.generation_time:.2f}s)")
del model
torch.cuda.empty_cache()

In [ ]:
# Cell 5: Run Vanilla baseline (all COCO splits)
!python scripts/run_7b_eval.py \
    --model llava-hf/llava-1.5-7b-hf \
    --image-dir data/coco/val2014 \
    --pope-dir data/pope \
    --split coco_random \
    --methods vanilla \
    --max-questions 3000 \
    --output-dir experiments/7b \
    --use-flash

In [ ]:
# Cell 6: Run CVG dual-pass (4-bit quantized for T4)
# Delete any old invalid results first
!rm -f experiments/7b/cvg_7b_coco_random.json
!python scripts/run_7b_eval.py \
    --model llava-hf/llava-1.5-7b-hf \
    --image-dir data/coco/val2014 \
    --pope-dir data/pope \
    --split coco_random \
    --methods cvg_7b \
    --max-questions 3000 \
    --output-dir experiments/7b \
    --quantize 4bit

In [ ]:
# Cell 7: Run CVG-E single-pass with hooks (4-bit quantized for T4)
# Delete any old invalid results first
!rm -f experiments/7b/cvg_e_7b_coco_random.json
!python scripts/run_7b_eval.py \
    --model llava-hf/llava-1.5-7b-hf \
    --image-dir data/coco/val2014 \
    --pope-dir data/pope \
    --split coco_random \
    --methods cvg_e_7b \
    --max-questions 3000 \
    --output-dir experiments/7b \
    --quantize 4bit

In [ ]:
# Cell 8: Run popular + adversarial splits for all methods
for split in ["coco_popular", "coco_adversarial"]:
    for method in ["vanilla", "cvg_7b", "cvg_e_7b"]:
        flash = "--use-flash" if method == "vanilla" else ""
        quantize = "--quantize 4bit" if method != "vanilla" else ""
        !python scripts/run_7b_eval.py \
            --model llava-hf/llava-1.5-7b-hf \
            --image-dir data/coco/val2014 \
            --pope-dir data/pope \
            --split {split} \
            --methods {method} \
            --max-questions 3000 \
            --output-dir experiments/7b {flash} {quantize}

In [ ]:
# Cell 9: Analyze results and generate comparison table
import json
from pathlib import Path

results_dir = Path("experiments/7b")
all_results = {}

for f in sorted(results_dir.glob("*.json")):
    with open(f) as fh:
        data = json.load(fh)
    name = f.stem
    all_results[name] = data

print(f"{'File':<35} {'Parse':>8} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'Yes%':>7} {'Time':>7}")
print("-" * 95)
for name, r in all_results.items():
    parse_str = f"{r['n_parseable']}/{r['n_attempted']}"
    print(f"{name:<35} {parse_str:>8} {r['accuracy']:7.1f} {r['precision']:7.1f} "
          f"{r['recall']:7.1f} {r['f1']:7.1f} {r['yes_ratio']:7.1f} {r['avg_time']:7.3f}s")

In [ ]:
# Cell 10: Generate LaTeX table rows for paper
print("\n% LaTeX table rows (Table 1: POPE COCO results)")
print("% Method & ran. & pop. & adv. & Avg \\\\")

for method in ["vanilla", "cvg_7b", "cvg_e_7b"]:
    accs = []
    for split in ["coco_random", "coco_popular", "coco_adversarial"]:
        key = f"{method}_{split}"
        if key in all_results:
            accs.append(all_results[key]["accuracy"])
        else:
            accs.append(0)
    avg = sum(accs) / len(accs) if accs else 0
    label = {"vanilla": "Vanilla", "cvg_7b": "CVG", "cvg_e_7b": "CVG-E"}.get(method, method)
    print(f"{label} & {accs[0]:.1f} & {accs[1]:.1f} & {accs[2]:.1f} & {avg:.1f} \\\\")

In [ ]:
# Cell 11: Latency comparison (the key CVG-E win)
print("\n% Latency comparison")
print(f"{'Method':<15} {'Time/q':>10} {'vs Vanilla':>12}")
print("-" * 40)

vanilla_time = None
for method in ["vanilla", "cvg_7b", "cvg_e_7b"]:
    key = f"{method}_coco_random"
    if key in all_results:
        t = all_results[key]["avg_time"]
        if method == "vanilla":
            vanilla_time = t
        ratio = t / vanilla_time if vanilla_time else 0
        label = {"vanilla": "Vanilla", "cvg_7b": "CVG", "cvg_e_7b": "CVG-E"}.get(method)
        print(f"{label:<15} {t:10.3f}s {ratio:11.1f}x")

In [ ]:
# Cell 12: Save all results to a single summary file
summary = {}
for name, r in all_results.items():
    summary[name] = {
        k: v for k, v in r.items() if k != "responses"
    }

with open("experiments/7b/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Summary saved to experiments/7b/summary.json")